In [4]:
import math
import random
import sqlite3
from pathlib import Path
from typing import Any


# =========================================================
# 설정
# =========================================================

# 로컬 환경
DB_PATH = Path("local.db")

# Colab에서 업로드된 DB를 직접 사용할 경우
# DB_PATH = Path("/content/local.db")

# 현재 대화에 업로드된 DB로 테스트할 경우
# DB_PATH = Path("/mnt/data/local(1).db")


# 요청 위치에서 응답 좌표를 생성할 거리 범위
MIN_RANDOM_DISTANCE_METERS = 50
MAX_RANDOM_DISTANCE_METERS = 500

# 자동 생성 응답의 비밀번호
DEFAULT_PASSWORD = "1234"

# 같은 request_id에 응답이 이미 있으면 건너뛸지 여부
SKIP_EXISTING_RESPONSE = True

# 동일한 시드를 사용하면 같은 랜덤 좌표가 생성됨
# 실행할 때마다 다른 좌표를 만들려면 None으로 변경
RANDOM_SEED = 42


# =========================================================
# DB 연결
# =========================================================

def get_connection() -> sqlite3.Connection:
    if not DB_PATH.exists():
        raise FileNotFoundError(
            f"DB 파일을 찾을 수 없습니다: {DB_PATH.resolve()}"
        )

    connection = sqlite3.connect(DB_PATH)
    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA foreign_keys = ON")

    return connection


# =========================================================
# 테이블 생성
# =========================================================

def create_community_response_table() -> None:
    """
    community_response 테이블이 없으면 생성한다.
    """

    connection = get_connection()

    try:
        connection.execute(
            """
            CREATE TABLE IF NOT EXISTS community_response (
                response_id INTEGER PRIMARY KEY AUTOINCREMENT,

                request_id INTEGER NOT NULL,

                facility_name TEXT NOT NULL,

                latitude REAL NOT NULL,
                longitude REAL NOT NULL,

                address TEXT,
                description TEXT,

                place_contentid TEXT,

                password TEXT NOT NULL,

                created_at TEXT NOT NULL
                    DEFAULT CURRENT_TIMESTAMP,

                updated_at TEXT NOT NULL
                    DEFAULT CURRENT_TIMESTAMP,

                FOREIGN KEY (request_id)
                    REFERENCES community_request(request_id)
                    ON DELETE CASCADE,

                FOREIGN KEY (place_contentid)
                    REFERENCES place(contentid)
                    ON DELETE SET NULL
            )
            """
        )

        connection.commit()

    finally:
        connection.close()


# =========================================================
# 두 좌표 사이 거리 계산
# =========================================================

def calculate_distance_meters(
    latitude1: float,
    longitude1: float,
    latitude2: float,
    longitude2: float,
) -> float:
    """
    Haversine 공식을 이용해 두 좌표 사이 거리를
    미터 단위로 계산한다.
    """

    earth_radius_meters = 6_371_000

    latitude1_radian = math.radians(latitude1)
    latitude2_radian = math.radians(latitude2)

    latitude_difference = math.radians(
        latitude2 - latitude1
    )

    longitude_difference = math.radians(
        longitude2 - longitude1
    )

    a = (
        math.sin(latitude_difference / 2) ** 2
        + math.cos(latitude1_radian)
        * math.cos(latitude2_radian)
        * math.sin(longitude_difference / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a),
    )

    return earth_radius_meters * c


# =========================================================
# 요청 위치 주변에 랜덤 좌표 생성
# =========================================================

def generate_random_coordinate(
    center_latitude: float,
    center_longitude: float,
    min_distance_meters: float,
    max_distance_meters: float,
) -> tuple[float, float]:
    """
    기준 위치에서 최소 거리 이상,
    최대 거리 이하인 랜덤 좌표를 생성한다.
    """

    if min_distance_meters < 0:
        raise ValueError(
            "최소 거리는 0 이상이어야 합니다."
        )

    if max_distance_meters <= 0:
        raise ValueError(
            "최대 거리는 0보다 커야 합니다."
        )

    if min_distance_meters > max_distance_meters:
        raise ValueError(
            "최소 거리가 최대 거리보다 클 수 없습니다."
        )

    earth_radius_meters = 6_371_000

    # 원 내부에 좌표가 균등하게 분포하도록 생성
    minimum_ratio = (
        min_distance_meters
        / max_distance_meters
    )

    random_distance = (
        max_distance_meters
        * math.sqrt(
            random.uniform(
                minimum_ratio ** 2,
                1,
            )
        )
    )

    # 0~360도 방향
    random_bearing = random.uniform(
        0,
        2 * math.pi,
    )

    angular_distance = (
        random_distance
        / earth_radius_meters
    )

    center_latitude_radian = math.radians(
        center_latitude
    )

    center_longitude_radian = math.radians(
        center_longitude
    )

    random_latitude_radian = math.asin(
        math.sin(center_latitude_radian)
        * math.cos(angular_distance)
        + math.cos(center_latitude_radian)
        * math.sin(angular_distance)
        * math.cos(random_bearing)
    )

    random_longitude_radian = (
        center_longitude_radian
        + math.atan2(
            math.sin(random_bearing)
            * math.sin(angular_distance)
            * math.cos(center_latitude_radian),

            math.cos(angular_distance)
            - math.sin(center_latitude_radian)
            * math.sin(random_latitude_radian),
        )
    )

    random_latitude = math.degrees(
        random_latitude_radian
    )

    random_longitude = math.degrees(
        random_longitude_radian
    )

    return (
        round(random_latitude, 7),
        round(random_longitude, 7),
    )


# =========================================================
# 랜덤 좌표에서 가장 가까운 place 찾기
# =========================================================

def find_nearest_place(
    latitude: float,
    longitude: float,
    places: list[sqlite3.Row],
) -> tuple[dict[str, Any] | None, float | None]:
    """
    전달받은 좌표에서 가장 가까운 place 한 개를 찾는다.
    """

    nearest_place = None
    nearest_distance = None

    for place_row in places:
        place_latitude = place_row["mapy"]
        place_longitude = place_row["mapx"]

        if (
            place_latitude is None
            or place_longitude is None
        ):
            continue

        distance_meters = calculate_distance_meters(
            latitude1=latitude,
            longitude1=longitude,
            latitude2=place_latitude,
            longitude2=place_longitude,
        )

        if (
            nearest_distance is None
            or distance_meters < nearest_distance
        ):
            nearest_place = dict(place_row)
            nearest_distance = distance_meters

    return nearest_place, nearest_distance


# =========================================================
# 모든 community_request에 응답 생성
# =========================================================

def create_responses_for_all_requests(
    min_random_distance_meters: float = 50,
    max_random_distance_meters: float = 500,
    default_password: str = "1234",
    skip_existing_response: bool = True,
    random_seed: int | None = 42,
) -> dict[str, Any]:
    """
    community_request의 모든 행을 읽고
    요청마다 community_response를 하나씩 생성한다.

    저장 방식:
    - request_id:
        원본 community_request의 request_id

    - latitude, longitude:
        요청 위치 근처에서 생성한 랜덤 좌표

    - facility_name:
        랜덤 좌표에서 가장 가까운 place.title

    - place_contentid:
        가장 가까운 place.contentid

    - address:
        가장 가까운 place.addr1
    """

    if random_seed is not None:
        random.seed(random_seed)

    connection = get_connection()

    inserted_count = 0
    skipped_count = 0
    failed_count = 0

    created_responses = []

    try:
        request_rows = connection.execute(
            """
            SELECT
                request_id,
                question,
                requested_facility,
                latitude,
                longitude,
                address
            FROM community_request
            WHERE latitude IS NOT NULL
              AND longitude IS NOT NULL
            ORDER BY request_id
            """
        ).fetchall()

        place_rows = connection.execute(
            """
            SELECT
                contentid,
                contenttypeid,
                title,
                addr1,
                firstimage,
                firstimage2,
                mapx,
                mapy
            FROM place
            WHERE title IS NOT NULL
              AND mapx IS NOT NULL
              AND mapy IS NOT NULL
            """
        ).fetchall()

        if len(request_rows) == 0:
            raise ValueError(
                "community_request 테이블에 처리할 행이 없습니다."
            )

        if len(place_rows) == 0:
            raise ValueError(
                "place 테이블에 위치 데이터가 없습니다."
            )

        for request_row in request_rows:
            request_id = request_row["request_id"]

            try:
                # -----------------------------------------
                # 기존 응답 확인
                # -----------------------------------------
                if skip_existing_response:
                    existing_response = connection.execute(
                        """
                        SELECT response_id
                        FROM community_response
                        WHERE request_id = ?
                        LIMIT 1
                        """,
                        (request_id,),
                    ).fetchone()

                    if existing_response is not None:
                        skipped_count += 1
                        continue

                request_latitude = float(
                    request_row["latitude"]
                )

                request_longitude = float(
                    request_row["longitude"]
                )

                # -----------------------------------------
                # 요청 위치 주변에 랜덤 좌표 생성
                # -----------------------------------------
                (
                    response_latitude,
                    response_longitude,
                ) = generate_random_coordinate(
                    center_latitude=request_latitude,
                    center_longitude=request_longitude,
                    min_distance_meters=(
                        min_random_distance_meters
                    ),
                    max_distance_meters=(
                        max_random_distance_meters
                    ),
                )

                # -----------------------------------------
                # 랜덤 좌표에서 가장 가까운 place 검색
                # -----------------------------------------
                (
                    nearest_place,
                    response_to_place_distance,
                ) = find_nearest_place(
                    latitude=response_latitude,
                    longitude=response_longitude,
                    places=place_rows,
                )

                if nearest_place is None:
                    failed_count += 1

                    print(
                        f"[실패] request_id={request_id}: "
                        "가까운 place를 찾지 못했습니다."
                    )

                    continue

                # 요청 위치와 랜덤 응답 위치 사이 거리
                request_to_response_distance = (
                    calculate_distance_meters(
                        latitude1=request_latitude,
                        longitude1=request_longitude,
                        latitude2=response_latitude,
                        longitude2=response_longitude,
                    )
                )

                requested_facility = (
                    request_row["requested_facility"]
                    or "시설"
                )

                facility_name = nearest_place["title"]

                description = (
                    f'{requested_facility} 제보 위치입니다. '
                    f'요청 위치에서 약 '
                    f'{request_to_response_distance:.1f}m '
                    f'떨어져 있습니다. '
                    f'이 좌표에서 가장 가까운 장소는 '
                    f'{facility_name}입니다.'
                )

                # -----------------------------------------
                # community_response 저장
                # -----------------------------------------
                cursor = connection.execute(
                    """
                    INSERT INTO community_response (
                        request_id,
                        facility_name,
                        latitude,
                        longitude,
                        address,
                        description,
                        place_contentid,
                        password
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                    """,
                    (
                        request_id,
                        facility_name,
                        response_latitude,
                        response_longitude,
                        nearest_place["addr1"],
                        description,
                        nearest_place["contentid"],
                        default_password,
                    ),
                )

                inserted_count += 1

                result_item = {
                    "response_id": cursor.lastrowid,
                    "request_id": request_id,
                    "requested_facility": requested_facility,
                    "request_latitude": request_latitude,
                    "request_longitude": request_longitude,
                    "response_latitude": response_latitude,
                    "response_longitude": response_longitude,
                    "facility_name": facility_name,
                    "place_contentid": nearest_place["contentid"],
                    "request_to_response_meters": round(
                        request_to_response_distance,
                        1,
                    ),
                    "response_to_place_meters": round(
                        response_to_place_distance,
                        1,
                    ),
                }

                created_responses.append(result_item)

                print(
                    f"[생성] request_id={request_id}, "
                    f"요청 시설={requested_facility}, "
                    f"facility_name={facility_name}, "
                    f"요청 거리="
                    f"{request_to_response_distance:.1f}m"
                )

            except Exception as error:
                failed_count += 1

                print(
                    f"[실패] request_id={request_id}: "
                    f"{error}"
                )

        connection.commit()

        return {
            "total_request_count": len(request_rows),
            "inserted_count": inserted_count,
            "skipped_count": skipped_count,
            "failed_count": failed_count,
            "created_responses": created_responses,
        }

    except Exception:
        connection.rollback()
        raise

    finally:
        connection.close()


# =========================================================
# 생성된 응답 확인
# =========================================================

def print_created_responses(
    limit: int = 10,
) -> None:
    connection = get_connection()

    try:
        rows = connection.execute(
            """
            SELECT
                response.response_id,
                response.request_id,

                request.requested_facility,

                request.latitude
                    AS request_latitude,

                request.longitude
                    AS request_longitude,

                response.facility_name,

                response.latitude
                    AS response_latitude,

                response.longitude
                    AS response_longitude,

                response.address,
                response.description,
                response.place_contentid,

                place.title
                    AS place_title,

                place.mapy
                    AS place_latitude,

                place.mapx
                    AS place_longitude,

                response.created_at,
                response.updated_at

            FROM community_response AS response

            JOIN community_request AS request
                ON request.request_id
                = response.request_id

            LEFT JOIN place
                ON place.contentid
                = response.place_contentid

            ORDER BY response.response_id

            LIMIT ?
            """,
            (limit,),
        ).fetchall()

        print()
        print("=" * 80)
        print(f"생성된 응답 중 {len(rows)}개 확인")
        print("=" * 80)

        for row in rows:
            print(dict(row))

    finally:
        connection.close()


# =========================================================
# 테이블 행 개수 확인
# =========================================================

def print_table_counts() -> None:
    connection = get_connection()

    try:
        request_count = connection.execute(
            """
            SELECT COUNT(*)
            FROM community_request
            """
        ).fetchone()[0]

        response_count = connection.execute(
            """
            SELECT COUNT(*)
            FROM community_response
            """
        ).fetchone()[0]

        place_count = connection.execute(
            """
            SELECT COUNT(*)
            FROM place
            """
        ).fetchone()[0]

        print()
        print("community_request:", request_count)
        print("community_response:", response_count)
        print("place:", place_count)

    finally:
        connection.close()


# =========================================================
# 기존 응답 전체 삭제
# =========================================================

def delete_all_community_responses() -> None:
    connection = get_connection()

    try:
        cursor = connection.execute(
            """
            DELETE FROM community_response
            """
        )

        connection.commit()

        print(
            f"community_response에서 "
            f"{cursor.rowcount}개 행을 삭제했습니다."
        )

    except Exception:
        connection.rollback()
        raise

    finally:
        connection.close()


# =========================================================
# 실행
# =========================================================

if __name__ == "__main__":
    # 응답 테이블이 없으면 생성
    create_community_response_table()

    # 실행 전 개수 확인
    print_table_counts()

    # 모든 요청에 응답 생성
    result = create_responses_for_all_requests(
        min_random_distance_meters=(
            MIN_RANDOM_DISTANCE_METERS
        ),
        max_random_distance_meters=(
            MAX_RANDOM_DISTANCE_METERS
        ),
        default_password=DEFAULT_PASSWORD,
        skip_existing_response=(
            SKIP_EXISTING_RESPONSE
        ),
        random_seed=RANDOM_SEED,
    )

    print()
    print("=" * 80)
    print("응답 생성 완료")
    print("=" * 80)

    print(
        "전체 요청 수:",
        result["total_request_count"],
    )

    print(
        "생성된 응답 수:",
        result["inserted_count"],
    )

    print(
        "기존 응답으로 건너뛴 수:",
        result["skipped_count"],
    )

    print(
        "실패한 수:",
        result["failed_count"],
    )

    # 실행 후 개수 확인
    print_table_counts()

    # 생성된 응답 10개 확인
    print_created_responses(limit=10)


community_request: 100
community_response: 0
place: 1365
[생성] request_id=1, 요청 시설=화장실, facility_name=트리플디, 요청 거리=400.9m
[생성] request_id=2, 요청 시설=철봉, facility_name=갈마공원, 요청 거리=265.7m
[생성] request_id=3, 요청 시설=주차장, facility_name=마치광장, 요청 거리=429.9m
[생성] request_id=4, 요청 시설=편의점, facility_name=바구스스파, 요청 거리=472.6m
[생성] request_id=5, 요청 시설=카페, facility_name=호텔 인터시티, 요청 거리=327.0m
[생성] request_id=6, 요청 시설=공원, facility_name=충남대학교박물관, 요청 거리=237.9m
[생성] request_id=7, 요청 시설=놀이터, facility_name=전미원, 요청 거리=95.2m
[생성] request_id=8, 요청 시설=운동시설, facility_name=논골어린이공원, 요청 거리=404.2m
[생성] request_id=9, 요청 시설=약국, facility_name=대전 중구문화원, 요청 거리=238.9m
[생성] request_id=10, 요청 시설=병원, facility_name=바다황제, 요청 거리=450.4m
[생성] request_id=11, 요청 시설=ATM, facility_name=LG전자 베스트샵 서대전점, 요청 거리=449.4m
[생성] request_id=12, 요청 시설=버스정류장, facility_name=유천시장, 요청 거리=294.5m
[생성] request_id=13, 요청 시설=자전거 거치대, facility_name=대전 동구문화원, 요청 거리=489.3m
[생성] request_id=14, 요청 시설=전기차 충전소, facility_name=롯데하이마트 용전점, 요청 거리=159.5m
[생성] request_id=